# 01. MCP (Model Context Protocol) with LangChain

> **주제**: 에이전트가 *외부 도구/데이터*에 표준 방식으로 접근하기 — `langchain-mcp-adapters`
>
> **원문**: https://docs.langchain.com/oss/python/langchain/mcp

---

## 이 노트북에서 배우는 것

1. MCP 가 무엇이고 왜 필요한가 (USB-C 비유)
2. `MultiServerMCPClient` 로 여러 MCP 서버의 도구를 한 번에 불러오기
3. `stdio` vs `http` 전송 방식
4. `FastMCP` 로 나만의 MCP 서버 만들기
5. 상태 유지 세션, 리소스/프롬프트, 인터셉터, 콜백, Elicitation

각 셀은 **개념 설명 → 실행 코드** 순서로 구성되어 있습니다.

## 1. MCP 한눈에 보기

**MCP(Model Context Protocol)** 는 LLM 애플리케이션이 외부 도구·데이터 소스에 연결하는 방식을 표준화한 오픈 프로토콜입니다.

> 흔히 **"AI 애플리케이션을 위한 USB-C"** 라고 비유합니다. 도구마다 제각각인 연결 방식을 하나의 규격으로 통일하는 것이죠.

LangChain 에서는 `langchain-mcp-adapters` 패키지가 MCP 서버가 노출하는 **tools / resources / prompts** 를 LangChain 의 `BaseTool` 객체로 변환해 줍니다. 그래서 한 번 변환하면 일반 LangChain 도구처럼 `create_agent` 에 바로 넘길 수 있습니다.

```
[LangChain Agent] --(MCP adapters)--> [MCP Server 1: math]
                                  \-> [MCP Server 2: weather]
```

In [46]:
# 설치 (uv)
!uv pip install -q python-dotenv langchain-mcp-adapters langchain langchain-openai "langchain[anthropic]"
# uv 프로젝트라면:  uv add langchain-mcp-adapters

# 직접 MCP 서버를 만들어 볼 때 추가로:
!uv pip install -q fastmcp

from dotenv import load_dotenv
load_dotenv("/Users/sehun/work/langchain-docs/deep-agents/week3-protocols-sehun/.env", override=True)

True

## 2. 먼저 MCP 서버 두 개를 만든다 (FastMCP)

클라이언트를 실험하려면 붙을 서버가 필요합니다. `FastMCP` 로 아주 작은 서버 두 개를 파일로 떨궈 둡니다.

- **math 서버**: `stdio` 전송 — 클라이언트가 서브프로세스로 실행
- **weather 서버**: `streamable-http` 전송 — 별도 포트에서 동작

`@mcp.tool()` 로 데코레이트된 함수가 그대로 도구가 되고, **docstring 이 도구 설명** 으로 쓰여 LLM 이 언제 호출할지 판단합니다.

In [47]:
# math_server.py 작성 (stdio)
math_server = '''
from fastmcp import FastMCP

mcp = FastMCP("Math")

@mcp.tool()
def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b

@mcp.tool()
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b

if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

with open("math_server.py", "w") as f:
    f.write(math_server)
print("math_server.py 생성 완료")

math_server.py 생성 완료


In [48]:
# weather_server.py 작성 (streamable-http)
weather_server = '''
from __future__ import annotations

import os

from fastmcp import FastMCP
from fastmcp.server.auth import AccessToken, TokenVerifier

EXPECTED_TOKEN = os.environ.get("MCP_SERVER_TOKEN", "demo-secret-token")
PORT = int(os.environ.get("MCP_WEATHER_PORT", "8765"))


class StaticTokenVerifier(TokenVerifier):
    """가장 단순한 토큰 검증기 — 사전 공유된 정적 토큰 하나와 비교한다.

    실전에서는 JWT 서명 검증(fastmcp.server.auth.providers.jwt)이나
    OAuth introspection 을 쓴다. 여기서는 인증 흐름을 눈으로 보기 위한
    학습용 구현이다.
    """

    def __init__(self, valid_token: str, **kwargs) -> None:
        super().__init__(**kwargs)
        self._valid_token = valid_token

    async def verify_token(self, token: str) -> AccessToken | None:
        if token == self._valid_token:
            # 유효 -> 이 토큰의 권한 정보를 반환하면 요청이 통과한다.
            return AccessToken(token=token, client_id="weather-demo-client", scopes=[])
        # None -> 401 Unauthorized 로 거부된다.
        return None


mcp = FastMCP("Weather", auth=StaticTokenVerifier(EXPECTED_TOKEN))


@mcp.tool()
async def get_weather(location: str) -> str:
    """Get weather for location."""
    return "It's always sunny in New York"


if __name__ == "__main__":
    print(f"Weather MCP 서버 시작 — http://127.0.0.1:{PORT}/mcp")
    print(f"기대 토큰: {EXPECTED_TOKEN!r}  (헤더: 'Authorization: Bearer {EXPECTED_TOKEN}')")
    mcp.run(transport="streamable-http", host="127.0.0.1", port=PORT)
'''
with open("weather_server.py", "w") as f:
    f.write(weather_server)
print("weather_server.py 생성 완료")
# weather 서버는 HTTP 이므로 백그라운드로 띄워 둔다 (터미널에서 실행해도 됨):
#   python weather_server.py   ->  http://localhost:8765/mcp

weather_server.py 생성 완료


## 3. `MultiServerMCPClient` — 여러 서버를 한 번에

핵심 클래스입니다. 서버 이름을 키로 갖는 딕셔너리로 **여러 MCP 서버**를 동시에 등록하고, `get_tools()` 한 번으로 모든 도구를 모읍니다.

전송 방식별 설정 항목:

| 전송 | 필수 키 | 설명 |
|------|---------|------|
| `stdio` | `command`, `args` | 로컬 서브프로세스로 서버 실행 |
| `http`  | `url` (옵션 `headers`) | 원격/로컬 HTTP 서버에 연결 |

> ⚠️ `MultiServerMCPClient` 는 **기본적으로 stateless** 입니다. 도구를 호출할 때마다 세션을 새로 열고 → 실행 → 정리합니다. 호출 간 상태가 필요하면 7장의 `client.session()` 을 쓰세요.

In [49]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent
import os

client = MultiServerMCPClient(
    {
        "math": {
            "transport": "stdio",
            "command": "python",
            "args": ["/Users/sehun/work/langchain-docs/deep-agents/week3-protocols-sehun/math_server.py"],  # 위에서 만든 파일 (절대경로 권장)
        },
        "weather": {
            "transport": "http",
            "headers": {
                "Authorization": f"Bearer {os.environ.get('MCP_SERVER_TOKEN', 'demo-secret-token')}"
            },
            "url": "http://localhost:8765/mcp",
        },
    }
)

# 모든 서버의 도구를 LangChain 도구로 로드
tools = await client.get_tools()
print("로드된 도구:", [t.name for t in tools])

로드된 도구: ['add', 'multiply', 'get_weather']


In [51]:
# 에이전트에 도구를 그대로 연결한다
# (OpenAI 모델 사용 — provider:model 형식. 키는 OPENAI_API_KEY 환경변수)
agent = create_agent("openai:gpt-4o-mini", tools) # 기존: "claude-sonnet-4-6"

# result = await agent.ainvoke(
#     {"messages": [{"role": "user", "content": "what's (3 + 5) x 12?"}]}
# )
result = await agent.ainvoke(
    {"messages": [{"role": "user", "content": "what's the weather in New York?"}]}
)
print(result["messages"][-1].content)

The weather in New York is sunny!


## 4. HTTP 전송 + 인증 헤더

원격 MCP 서버는 보통 토큰 인증을 요구합니다. `headers` 로 `Authorization` 을 넘깁니다.

In [52]:
client = MultiServerMCPClient(
    {
        "weather": {
            "transport": "http",
            "url": "http://localhost:8765/mcp",
            "headers": {"Authorization": "Bearer 123123123"},
        }
    }
)
tools = await client.get_tools() # token 인증 오류 유도

  + Exception Group Traceback (most recent call last):
  |   File "/Users/sehun/work/langchain-docs/deep-agents/week3-protocols-sehun/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py", line 3746, in run_code
  |     await eval(code_obj, self.user_global_ns, self.user_ns)
  |   File "/var/folders/n0/szcdn28x72l1lxd9b3xzjv340000gn/T/ipykernel_34700/3463644389.py", line 10, in <module>
  |     tools = await client.get_tools() # token 인증 오류 유도
  |             ^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "/Users/sehun/work/langchain-docs/deep-agents/week3-protocols-sehun/.venv/lib/python3.12/site-packages/langchain_mcp_adapters/client.py", line 197, in get_tools
  |     tools_list = await asyncio.gather(*load_mcp_tool_tasks)
  |                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "/Users/sehun/work/langchain-docs/deep-agents/week3-protocols-sehun/.venv/lib/python3.12/site-packages/langchain_mcp_adapters/tools.py", line 478, in load_mcp_tools
  |     async with c

## 5. 상태 유지 세션 (`client.session()`) — 직접 확인

`MultiServerMCPClient` 는 **기본이 stateless** 입니다. 도구를 호출할 때마다 세션을 새로 열고 → 실행 → 정리하죠. stdio 서버라면 **호출마다 서버 프로세스가 새로 뜨고 죽어서**, 서버가 메모리에 들고 있던 상태도 같이 사라집니다.

반대로 `async with client.session(...)` 블록 안에서는 **하나의 연결(프로세스)을 열어 둔 채** 여러 도구 호출이 같은 세션을 공유합니다. 그래서 서버측 상태가 호출 사이에 유지됩니다. (예: 로그인 세션, 커서 위치, 누적 카운터 등)

아래 데모는 **서버에 카운터를 두고** 두 방식을 비교합니다.

| 방식 | `increment()` 3번 호출 결과 | 이유 |
|------|------------------------------|------|
| stateless (기본 `get_tools()`) | `1, 1, 1` | 매 호출마다 새 프로세스 → 카운터 리셋 |
| stateful (`client.session()`) | `1, 2, 3` | 같은 프로세스 유지 → 카운터 누적 |

> 즉 같은 서버·같은 도구라도, **세션을 유지하느냐**에 따라 결과가 달라집니다.

In [27]:
# §5. 세션 유지(stateful) vs 기본(stateless) 비교 — 서버에 카운터를 두고 확인
import sys
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_mcp_adapters.tools import load_mcp_tools

# 상태(카운터)를 메모리에 들고 있는 MCP 서버를 파일로 생성
counter_server = '''
from fastmcp import FastMCP

mcp = FastMCP("Counter")
_state = {"n": 0}

@mcp.tool()
def increment() -> int:
    """Increment the server-side counter and return the new value."""
    _state["n"] += 1
    return _state["n"]

@mcp.tool()
def get_count() -> int:
    """Return the current counter value without changing it."""
    return _state["n"]

if __name__ == "__main__":
    mcp.run(transport="stdio")
'''
with open("counter_server.py", "w") as f:
    f.write(counter_server)

client = MultiServerMCPClient({
    "counter": {
        "transport": "stdio",
        "command": sys.executable,   # 활성 커널의 python (PATH 의존 X)
        "args": ["counter_server.py"],
    }
})


def value_of(result):
    """MCP 도구 결과(content block 리스트)에서 텍스트 값만 뽑는다."""
    if isinstance(result, list) and result and isinstance(result[0], dict):
        return result[0].get("text")
    return result


# (1) STATELESS — 기본 동작: 호출마다 새 세션(새 프로세스)
tools = await client.get_tools()
inc = {t.name: t for t in tools}["increment"]
stateless = [value_of(await inc.ainvoke({})) for _ in range(3)]
print("stateless (기본 get_tools)  increment x3 ->", stateless)

# (2) STATEFUL — client.session(): 한 세션(프로세스)을 유지
async with client.session("counter") as session:
    s_tools = await load_mcp_tools(session)
    s_inc = {t.name: t for t in s_tools}["increment"]
    stateful = [value_of(await s_inc.ainvoke({})) for _ in range(3)]
print("stateful  (client.session)  increment x3 ->", stateful)

print("\n→ stateless 는 매번 1 (프로세스가 새로 떠 카운터 리셋),")
print("  stateful 은 1\u00b72\u00b73 (같은 프로세스라 카운터 누적)")

stateless (기본 get_tools)  increment x3 -> ['1', '1', '1']
stateful  (client.session)  increment x3 -> ['1', '2', '3']

→ stateless 는 매번 1 (프로세스가 새로 떠 카운터 리셋),
  stateful 은 1·2·3 (같은 프로세스라 카운터 누적)


## 6. 구조화된 결과 / 리소스 / 프롬프트

MCP 도구는 텍스트뿐 아니라 **구조화된 콘텐츠**를 반환할 수 있습니다. 결과는 `ToolMessage.artifact` 에 담깁니다.

또한 서버는 도구 외에도 **resources**(읽을 수 있는 데이터 blob)와 **prompts**(재사용 프롬프트 템플릿)를 노출할 수 있습니다.

In [31]:
from langchain.messages import ToolMessage

result = await agent.ainvoke(
    {"messages": [{"role": "user", "content": "what's (3 + 5) x 12?"}]}
)

for message in result["messages"]:
    if isinstance(message, ToolMessage) and message.artifact:
        structured_content = message.artifact["structured_content"]
        print(f"message.artifact: {message.artifact}")
        print(structured_content)

message.artifact: {'structured_content': {'result': 8}}
{'result': 8}
message.artifact: {'structured_content': {'result': 96}}
{'result': 96}


In [33]:
# §6-1. 리소스 로드 — 리소스를 노출하는 서버를 만들어 실제 결과 확인
# (math 서버에는 리소스가 없어서 'server_name' 호출은 실패한다.
#  여기서는 @mcp.resource 로 리소스를 노출하는 docs 서버를 띄운다.)
import sys
from langchain_mcp_adapters.client import MultiServerMCPClient

# resources 를 노출하는 MCP 서버를 파일로 생성
docs_server = '''
from fastmcp import FastMCP

mcp = FastMCP("Docs")

@mcp.resource("docs://readme")
def readme() -> str:
    """프로젝트 README 문서."""
    return "Week3 Protocols 실습 자료 — MCP/ACP/A2A"

@mcp.resource("config://settings")
def settings() -> str:
    """앱 설정(JSON)."""
    return '{"theme": "dark", "lang": "ko"}'

if __name__ == "__main__":
    mcp.run(transport="stdio")
'''
with open("docs_server.py", "w") as f:
    f.write(docs_server)

docs_client = MultiServerMCPClient({
    "docs": {
        "transport": "stdio",
        "command": sys.executable,   # 활성 커널의 python
        "args": ["docs_server.py"],
    }
})

# 서버가 노출하는 모든 리소스를 Blob 으로 로드
blobs = await docs_client.get_resources("docs")
print(f"로드된 리소스: {len(blobs)} 개\n")
for blob in blobs:
    print(f"URI : {blob.metadata['uri']}")
    print(f"MIME: {blob.mimetype}")
    print(f"내용: {blob.as_string()}")
    print("---")

# 특정 URI 만 골라서 로드할 수도 있다
only_readme = await docs_client.get_resources("docs", uris=["docs://readme"])
print("\n[URI 지정 로드] docs://readme ->", only_readme[0].as_string())

로드된 리소스: 2 개

URI : docs://readme
MIME: text/plain
내용: Week3 Protocols 실습 자료 — MCP/ACP/A2A
---
URI : config://settings
MIME: text/plain
내용: {"theme": "dark", "lang": "ko"}
---

[URI 지정 로드] docs://readme -> Week3 Protocols 실습 자료 — MCP/ACP/A2A


In [35]:
# §6-2. 프롬프트 로드 — @mcp.prompt 로 프롬프트를 노출하는 서버를 만들어 실제 결과 확인
# (math 서버에는 프롬프트가 없어서 'server_name' 호출은 실패한다.)
import sys
from langchain_mcp_adapters.client import MultiServerMCPClient

# prompts 를 노출하는 MCP 서버를 파일로 생성
prompts_server = '''
from fastmcp import FastMCP

mcp = FastMCP("Prompts")

@mcp.prompt
def summarize() -> str:
    """재사용 요약 프롬프트."""
    return "다음 내용을 핵심만 3문장으로 요약해줘."

@mcp.prompt
def code_review(language: str, focus: str) -> str:
    """언어/관점을 인자로 받는 코드 리뷰 프롬프트."""
    return f"다음 {language} 코드를 {focus} 관점에서 리뷰하고 개선점을 알려줘."

if __name__ == "__main__":
    mcp.run(transport="stdio")
'''
with open("prompts_server.py", "w") as f:
    f.write(prompts_server)

prompts_client = MultiServerMCPClient({
    "prompts": {
        "transport": "stdio",
        "command": sys.executable,   # 활성 커널의 python
        "args": ["prompts_server.py"],
    }
})

# 이름으로 프롬프트 로드 (인자 없음) — LangChain 메시지 리스트로 변환된다
messages = await prompts_client.get_prompt("prompts", "summarize")
print("[summarize]")
for m in messages:
    print(f"  {m.type}: {m.content}")

# 인자가 있는 프롬프트 로드
messages = await prompts_client.get_prompt(
    "prompts",
    "code_review",
    arguments={"language": "python", "focus": "security"},
)
print("\n[code_review(language=python, focus=security)]")
for m in messages:
    print(f"  {m.type}: {m.content}")

[summarize]
  human: 다음 내용을 핵심만 3문장으로 요약해줘.

[code_review(language=python, focus=security)]
  human: 다음 python 코드를 security 관점에서 리뷰하고 개선점을 알려줘.


## 7. 도구 인터셉터 — 호출 전후 가로채기

인터셉터는 미들웨어처럼 동작합니다. `handler` 를 호출하기 **전/후** 로 로깅·인자 변형·런타임 컨텍스트 주입 등을 할 수 있습니다. 아래 두 셀은 인터셉터가 실제로 발동하는 모습을 `print` 로 보여줍니다.

- **7-1**: 인터셉터 2개의 "양파(onion)" 실행 순서 (바깥 → 안 → 도구 → 안 → 바깥)
- **7-2**: 인자 변형(2배)과 런타임 컨텍스트 주입(`user_id` 자동 첨부)

In [39]:
# §7-1. 인터셉터 발동 순서 — "양파(onion)" 구조를 print 로 확인
import sys
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_mcp_adapters.interceptors import MCPToolCallRequest

# 도구 하나(multiply)를 가진 MCP 서버 생성
math_server = '''
from fastmcp import FastMCP
mcp = FastMCP("Math")
@mcp.tool()
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b
if __name__ == "__main__":
    mcp.run(transport="stdio")
'''
with open("math_intercept_server.py", "w") as f:
    f.write(math_server)


def val(r):
    return r[0]["text"] if isinstance(r, list) and r and isinstance(r[0], dict) else r


# 인터셉터 두 개 — 리스트 순서대로 '바깥 -> 안' 으로 감싼다
async def outer_interceptor(request: MCPToolCallRequest, handler):
    print(f"[outer] ▶ 호출 전  name={request.name} args={request.args}")
    result = await handler(request)
    print(f"[outer] ◀ 호출 후")
    return result

async def inner_interceptor(request: MCPToolCallRequest, handler):
    print(f"  [inner] ▶ 호출 전")
    result = await handler(request)
    print(f"  [inner] ◀ 호출 후")
    return result

client = MultiServerMCPClient(
    {"math": {"transport": "stdio", "command": sys.executable, "args": ["math_intercept_server.py"]}},
    # {"englsih": {"transport": "stdio", "command": sys.executable, "args": ["english_intercept_server.py"]}}
    tool_interceptors=[outer_interceptor, inner_interceptor],  # 첫 번째가 가장 바깥
)

tools = {t.name: t for t in await client.get_tools()}
result = await tools["multiply"].ainvoke({"a": 2, "b": 3})
print("결과:", val(result))
print("\n→ 실행 순서: outer 전 → inner 전 → (도구 실행) → inner 후 → outer 후")

[outer] ▶ 호출 전  name=multiply args={'a': 2, 'b': 3}
  [inner] ▶ 호출 전
  [inner] ◀ 호출 후
[outer] ◀ 호출 후
결과: 6

→ 실행 순서: outer 전 → inner 전 → (도구 실행) → inner 후 → outer 후


In [40]:
# §7-2. 인자 변형 & 런타임 컨텍스트 주입 — 인터셉터가 args 를 바꾸는 걸 print 로 확인
import sys
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_mcp_adapters.interceptors import MCPToolCallRequest

# multiply 와 whoami 를 가진 서버
srv = '''
from fastmcp import FastMCP
mcp = FastMCP("Math2")
@mcp.tool()
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b
@mcp.tool()
def whoami(user_id: str = "anonymous") -> str:
    """현재 user_id 를 알려준다."""
    return f"호출자 user_id = {user_id}"
if __name__ == "__main__":
    mcp.run(transport="stdio")
'''
with open("math2_server.py", "w") as f:
    f.write(srv)

CONN = {"m": {"transport": "stdio", "command": sys.executable, "args": ["math2_server.py"]}}


def val(r):
    return r[0]["text"] if isinstance(r, list) and r and isinstance(r[0], dict) else r


# (A) 인자 변형: 모든 숫자 인자를 2배로 바꿔서 서버에 전달
async def double_args(request: MCPToolCallRequest, handler):
    modified = {k: v * 2 for k, v in request.args.items()}
    print(f"[double] {request.name}: {request.args} -> {modified}")
    return await handler(request.override(args=modified))

client_a = MultiServerMCPClient(CONN, tool_interceptors=[double_args])
mul = {t.name: t for t in await client_a.get_tools()}["multiply"]
print("multiply(2, 3) 결과:", val(await mul.ainvoke({"a": 2, "b": 3})), " ← 인자가 2배(4×6)로 바뀌어 24")

print()

# (B) 런타임 컨텍스트 주입: 호출 인자에 user_id 자동 첨부
#     실전에서는 request.runtime.context.user_id 에서 꺼낸다(에이전트 컨텍스트).
#     여기서는 데모용으로 고정값을 주입한다.
async def inject_user_id(request: MCPToolCallRequest, handler):
    injected = {**request.args, "user_id": "user_123"}
    print(f"[inject] {request.name}: user_id 자동 첨부 -> {injected}")
    return await handler(request.override(args=injected))

client_b = MultiServerMCPClient(CONN, tool_interceptors=[inject_user_id])
who = {t.name: t for t in await client_b.get_tools()}["whoami"]
print("whoami() 결과:", val(await who.ainvoke({})), " ← 인자 없이 불렀지만 user_id 가 주입됨")

[double] multiply: {'a': 2, 'b': 3} -> {'a': 4, 'b': 6}
multiply(2, 3) 결과: 24  ← 인자가 2배(4×6)로 바뀌어 24

[inject] whoami: user_id 자동 첨부 -> {'user_id': 'user_123'}
whoami() 결과: 호출자 user_id = user_123  ← 인자 없이 불렀지만 user_id 가 주입됨


## 8. 진행률/로그 콜백 & Elicitation

콜백은 도구 실행 중 서버가 보내는 이벤트를 클라이언트가 받는 통로입니다. 아래 두 셀은 실제 서버를 띄워 콜백이 발동하는 것을 보여줍니다.

- **8-1**: 장시간 도구의 진행률(`on_progress`)을 `print` 로 수신
- **8-2**: 실행 도중 서버가 사용자 입력을 요청하는 **Elicitation**(`on_elicitation`). 셀을 실행하면 **입력창이 떠서, 내가 직접 값을 입력한 뒤에야 코드가 진행**됩니다.

In [41]:
# §8-1. 진행률 콜백 — 서버가 보고하는 진행률(on_progress)을 print 로 수신
import sys
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_mcp_adapters.callbacks import Callbacks, CallbackContext

# 단계별로 진행률을 보고하는 long_task 도구를 가진 서버
srv = '''
import asyncio
from fastmcp import FastMCP
from fastmcp.server.context import Context
mcp = FastMCP("Tasks")
@mcp.tool()
async def long_task(steps: int, ctx: Context) -> str:
    """단계별로 진행률을 보고하는 작업."""
    for i in range(1, steps + 1):
        await ctx.report_progress(progress=i, total=steps, message=f"step {i}")
        await asyncio.sleep(0.05)
    return f"{steps}단계 완료"
if __name__ == "__main__":
    mcp.run(transport="stdio")
'''
with open("tasks_server.py", "w") as f:
    f.write(srv)


def val(r):
    return r[0]["text"] if isinstance(r, list) and r and isinstance(r[0], dict) else r


# 진행률이 올 때마다 호출되는 콜백
async def on_progress(progress, total, message, context: CallbackContext):
    pct = (progress / total * 100) if total else progress
    print(f"[progress] {context.server_name}: {pct:.0f}% - {message}")

client = MultiServerMCPClient(
    {"tasks": {"transport": "stdio", "command": sys.executable, "args": ["tasks_server.py"]}},
    callbacks=Callbacks(on_progress=on_progress),
)
task = {t.name: t for t in await client.get_tools()}["long_task"]
result = await task.ainvoke({"steps": 4})
print("결과:", val(result))

[progress] tasks: 25% - step 1
[progress] tasks: 50% - step 2
[progress] tasks: 75% - step 3
[progress] tasks: 100% - step 4
결과: 4단계 완료


### Elicitation 호출 순서 (셀 실행 시)

[준비]
1. profile_server.py 파일 생성
2. MultiServerMCPClient(..., callbacks=Callbacks(on_elicitation=on_elicitation)) 생성
3. await client.get_tools()         → 서버 서브프로세스 기동 + create_profile 도구 로드

[본 실행]  await tool.ainvoke({"name": "Sehun"})
그 다음부터가 핑퐁입니다:


클라이언트(노트북)                         서버(profile_server.py)
─────────────────────────────────────────────────────────────
 ① tools/call create_profile(name=Sehun) ──▶
                                            ② create_profile 실행 시작
                                            ③ await ctx.elicit(...) 도달
                                               → 실행을 "멈추고" 입력 요청 전송
                              ◀── elicitation/create  ("Sehun 의 정보를…")
 ④ on_elicitation(params) 콜백 발동
    print("[elicit] 서버 요청…")
 ⑤ email = input(...)   ← 여기서 셀이 멈춤 (사용자 입력 대기)
 ⑥ age   = input(...)
 ⑦ return ElicitResult(action=accept/decline/cancel)
                              ── 응답 ──▶
                                            ⑧ ctx.elicit 가 결과 반환,
                                               create_profile 이어서 실행
                                            ⑨ return "…프로필 생성: …"
                              ◀── tools/call 결과 ──
 ⑩ tool.ainvoke 반환
 ⑪ print("결과:", …)

핵심은 당신이 on_elicitation을 호출하는 게 아니라, ClientSession의 백그라운드 수신 루프가 서버 요청을 받으면 대신 호출한다는 점입니다 (제어의 역전).
설치된 라이브러리 코드로 배선을 따라가 보면 3개 레이어로 연결됩니다.

## 레이어 1 — 등록 시 래핑 (langchain_mcp_adapters)
Callbacks(on_elicitation=fn) 를 넘기면 to_mcp_format() 이 내 함수를 MCP SDK가 기대하는 시그니처로 감쌉니다.
CallbackContext 는 클로저로 캡처해서 빼고, (mcp_context, params) 만 받는 형태로 변환합니다.

langchain_mcp_adapters/callbacks.py:128


async def mcp_elicitation_callback(mcp_context, params):
    return await on_elicitation(mcp_context, params, context)  # ← 내 함수 호출
이게 _MCPCallbacks.elicitation_callback 에 담깁니다.

## 레이어 2 — 세션 생성 인자로 전달 (langchain → mcp SDK)
sessions.py:433 에서 그 콜백을 ClientSession 생성 kwargs 로 넘깁니다.


params["session_kwargs"]["elicitation_callback"] = mcp_callbacks.elicitation_callback
mcp/client/session.py:137 에서 세션이 그걸 멤버로 저장합니다.


self._elicitation_callback = elicitation_callback or _default_elicitation_callback
여기서 끝. 이 시점엔 아직 아무도 안 불렀고, "서버가 물어오면 이걸 써라"고 등록만 된 상태입니다.

## 레이어 3 — 백그라운드 수신 루프가 디스패치 (mcp SDK)
get_tools()/client.session() 이 세션을 열 때, BaseSession 이 수신 루프를 백그라운드 태스크로 띄웁니다.

mcp/shared/session.py:224


self._task_group.start_soon(self._receive_loop)
이 루프는 서버 stdout 을 계속 읽다가, 들어온 메시지가 응답이 아니라 요청(id+method 보유)이면 RequestResponder 로 감싸 _received_request 로 넘깁니다.

mcp/shared/session.py:351


async for message in self._read_stream:
    ...
    responder = RequestResponder(...)
    await self._received_request(responder)
_received_request 가 JSON-RPC method 이름으로 분기합니다. elicitation/create 요청이면 ElicitRequest 케이스로 가서, 등록해 둔 콜백을 호출하고 그 반환값을 서버로 응답합니다.

mcp/client/session.py:571


case types.ElicitRequest(params=params):
    with responder:
        response = await self._elicitation_callback(ctx, params)  # ← 레이어1 래퍼 → 내 on_elicitation
        client_response = ClientResponse.validate_python(response)
        await responder.respond(client_response)                 # ← ElicitResult 를 서버로 회신

In [53]:
# §8-2. Elicitation — 셀을 실행하면 입력창이 떠서, 내가 입력한 값으로 코드가 진행된다
#   ⚠️ 이 셀은 실행 도중 input() 입력창에서 멈춘다. 이메일·나이를 입력해야 다음으로 진행된다.
#      (JupyterLab/VS Code 노트북 모두 셀 위에 입력 칸이 나타난다.)
import sys
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_mcp_adapters.callbacks import Callbacks
from mcp.types import ElicitResult

# 실행 도중 ctx.elicit 로 추가 정보를 요청하는 서버
srv = '''
from pydantic import BaseModel
from fastmcp import FastMCP
from fastmcp.server.context import Context
mcp = FastMCP("Profile")
class UserDetails(BaseModel):
    email: str
    age: int
@mcp.tool()
async def create_profile(name: str, ctx: Context) -> str:
    """elicitation 으로 이메일/나이를 받아 프로필 생성."""
    result = await ctx.elicit(message=f"{name} 의 정보를 입력하세요:", response_type=UserDetails)
    if result.action == "accept":
        return f"{name} 프로필 생성: email={result.data.email}, age={result.data.age}"
    if result.action == "decline":
        return f"{name}: 사용자가 입력을 거절함"
    return f"{name}: 작업 취소됨"
if __name__ == "__main__":
    mcp.run(transport="stdio")
'''
with open("profile_server.py", "w") as f:
    f.write(srv)


def val(r):
    return r[0]["text"] if isinstance(r, list) and r and isinstance(r[0], dict) else r


# 서버가 입력을 요청하면 -> 여기서 실제 사용자에게 input() 으로 물어본다.
# 세 가지 응답: accept(값 제공) / decline(거절) / cancel(취소)
async def on_elicitation(mcp_context, params, context):
    print(f"[elicit] 서버 요청: {params.message}")
    email = input("  이메일 입력 (빈 칸이면 거절): ").strip()
    if not email:
        print("[elicit] -> decline")
        return ElicitResult(action="decline")
    age_str = input("  나이 입력 (숫자가 아니면 취소): ").strip()
    if not age_str.isdigit():
        print("[elicit] -> cancel")
        return ElicitResult(action="cancel")
    print(f"[elicit] -> accept (email={email}, age={age_str})")
    return ElicitResult(action="accept", content={"email": email, "age": int(age_str)})

client = MultiServerMCPClient(
    {"profile": {"transport": "stdio", "command": sys.executable, "args": ["profile_server.py"]}},
    callbacks=Callbacks(on_elicitation=on_elicitation),
)
tool = {t.name: t for t in await client.get_tools()}["create_profile"]
print("create_profile 실행 중... (입력창이 뜨면 값을 입력하세요)")
result = await tool.ainvoke({"name": "Sehun"})
print("결과:", val(result))

create_profile 실행 중... (입력창이 뜨면 값을 입력하세요)
[elicit] 서버 요청: Sehun 의 정보를 입력하세요:


  이메일 입력 (빈 칸이면 거절):  123@naver.com
  나이 입력 (숫자가 아니면 취소):  23


[elicit] -> accept (email=123@naver.com, age=23)
결과: Sehun 프로필 생성: email=123@naver.com, age=23


## 정리 & 연습 문제

**핵심 요약**
- MCP = 도구/데이터 연결의 표준 규격. `langchain-mcp-adapters` 가 MCP → LangChain 도구로 변환
- `MultiServerMCPClient({이름: {transport, ...}})` → `get_tools()` → `create_agent`
- 전송: `stdio`(로컬 프로세스) / `http`(원격)
- 기본 stateless, 필요하면 `client.session()`
- 고급: 구조화 결과(`artifact`), resources/prompts, 인터셉터, 콜백, elicitation

**연습**
1. `math_server.py` 에 `subtract` 도구를 추가하고 에이전트로 `(20 - 8) x 3` 을 계산시켜 보세요.
2. `logging_interceptor` 를 실제 클라이언트에 연결해 도구 호출 로그를 출력해 보세요.
3. weather 서버를 `http` 로 띄우고 "What's the weather in New York?" 을 물어보세요.